In [1]:
# Import
import torch
import torch.nn as nn
import numpy as np
from tqdm.notebook import trange, tqdm
import h5py
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
%matplotlib inline
import ecg_plot, pandas as pd
import sys, glob, h5py
import ecgprep
import os
from imblearn.over_sampling import RandomOverSampler

In [12]:
!pwd

/mimer/NOBACKUP/groups/naiss2024-5-153/Aditya/2-soa-ecg-model


In [46]:
#!rm data/code15-12l/exams_part0.hdf5
#!unzip data/code15-12l/exams_part0.zip
subsets = glob.glob('data/code15-12l/*.hdf5')
PATH_TO_CSV = 'data/exams.csv'

#traces_ids = np.zeros(len(subsets)*29000) #                     >>>>>>>>> in resampling, keeping track of ids becomes problematic
labels_all = np.zeros(len(subsets)*29000)
layout = np.zeros((len(subsets)*29000, 4096, 12), dtype='float32') # layout = h5py.VirtualLayout(shape=(len(subsets)*29000, 4096, 12), dtype='float32')
subsets

['data/code15-12l/exams_part6.hdf5',
 'data/code15-12l/exams_part8.hdf5',
 'data/code15-12l/exams_part7.hdf5',
 'data/code15-12l/exams_part1.hdf5',
 'data/code15-12l/exams_part12.hdf5',
 'data/code15-12l/exams_part15.hdf5',
 'data/code15-12l/exams_part16.hdf5',
 'data/code15-12l/exams_part10.hdf5',
 'data/code15-12l/exams_part0.hdf5',
 'data/code15-12l/exams_part2.hdf5',
 'data/code15-12l/exams_part3.hdf5',
 'data/code15-12l/exams_part11.hdf5',
 'data/code15-12l/exams_part17.hdf5',
 'data/code15-12l/exams_part4.hdf5',
 'data/code15-12l/exams_part13.hdf5',
 'data/code15-12l/exams_part5.hdf5',
 'data/code15-12l/exams_part9.hdf5',
 'data/code15-12l/exams_part14.hdf5']

In [47]:
#!rm data/code15-12l/exams_part*.hdf5
#!cd data/code15-12l; unzip "*.zip"

In [48]:
%%time
prev = 0
for i, part in enumerate(subsets):
    try:
        if '-upsmpl' not in part:
            f = h5py.File(part, 'r')
            ecgchunk = np.array(f['tracings'][()], dtype=np.float32)
            idchunk = np.array(f['exam_id'])
            df = pd.read_csv(PATH_TO_CSV)
            idx = [i in idchunk for i in df['exam_id']]
            df = pd.DataFrame(df.loc[idx])
            df.set_index('exam_id', inplace=True)
            df = df.reindex(idchunk).dropna(subset=['AF'])
            
            if len(ecgchunk.reshape(len(ecgchunk), 4096*12)) - len(list(df['AF'])) > 0:
                difference = len(ecgchunk.reshape(len(ecgchunk), 4096*12)) - len(list(df['AF']))
                print("difference >>", difference)
                ecgchunk = np.delete(ecgchunk, -1, 0) # after re-indexing
            
            f.close()
            
            rus = RandomOverSampler(sampling_strategy=0.5, random_state=42)
            ecgchunk, labels = rus.fit_resample(ecgchunk.reshape(len(ecgchunk), 4096*12), list(df['AF']))
            
            print(ecgchunk.shape, len(labels), "<<<<<<<<< THIS SHOULDNT BE DIFFERENT")
            print("currently filled until >>", i*prev)
            resampled_ecgchunk = ecgchunk.reshape(len(ecgchunk),4096,12)[:29000,:,:]
            
            # ===================================================================== #
            with h5py.File(f'{part.split('.hdf5')[0]}-upsmpl.hdf5', "w") as fw:
                fw.create_dataset("tracings", data=resampled_ecgchunk) #f.create_virtual_dataset("tracings", layout)
                os.remove(part)
            # ===================================================================== #
            
            #layout[i*prev:(i+1)*ecgchunk.shape[0],:,:] = h5py.VirtualSource(part, "tracings", shape=tuple(ecgchunk.shape))
            labels_all[i*prev:(i+1)*resampled_ecgchunk.shape[0]] = labels[:29000] #        >>>>>>>>> maintaining the order is very important, as ids are challenging to keep track of during oversampling
            #traces_ids[i*prev:(i+1)*ecgchunk.size(dim=0)] = idchunk #                     >>>>>>>>> in resampling, keeping track of ids becomes problematic
            
            prev = int(resampled_ecgchunk.shape[0])
            print("filled values from idx ", i*prev, "to", (i+1)*resampled_ecgchunk.shape[0])
            print("Done! At number --", i)
    except:
        break

pd.DataFrame(labels_all).to_csv('data/exams-upsmplCODE15.csv', index=False)

difference >> 1
(29422, 49152) 29422 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 0
filled values from idx  0 to 29000
Done! At number -- 0
difference >> 1
(29433, 49152) 29433 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 29000
filled values from idx  29000 to 58000
Done! At number -- 1
difference >> 1
(29430, 49152) 29430 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 58000
filled values from idx  58000 to 87000
Done! At number -- 2
difference >> 1
(29448, 49152) 29448 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 87000
filled values from idx  87000 to 116000
Done! At number -- 3
difference >> 1
(29392, 49152) 29392 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 116000
filled values from idx  116000 to 145000
Done! At number -- 4
difference >> 1
(29394, 49152) 29394 <<<<<<<<< THIS SHOULDNT BE DIFFERENT
currently filled until >> 145000
filled values from idx  145000 to 174000
Done! At number -- 5
diffe

In [53]:
upsampled_files = glob.glob('data/code15-12l/*-upsmpl.hdf5')
len(upsampled_files[:-1])

12

In [59]:
# if the process above fails, adjust the shape of the virtual layout accordingly
# (348000,4096,12) - failed files --> 5,9,14
pd.DataFrame(labels_all[:348000]).to_csv('data/exams-upsmplCODE15.csv', index=False, header=False)
layout = h5py.VirtualLayout(shape=(len(upsampled_files)*29000, 4096, 12), dtype='float32')

prev = 0
for i, f in enumerate(upsampled_files[:-1]):
    print("filled values from idx ", i*prev, "to", (i+1)*29000)
    layout[i*prev:(i+1)*29000,:,:] = h5py.VirtualSource(f, "tracings", shape=(29000, 4096, 12))
    prev = 29000
    
with h5py.File('data/VDS-code15-upsmpl.hdf5', "w") as fw:
    fw.create_virtual_dataset("tracings", layout, fillvalue=-1)

filled values from idx  0 to 29000
filled values from idx  29000 to 58000
filled values from idx  58000 to 87000
filled values from idx  87000 to 116000
filled values from idx  116000 to 145000
filled values from idx  145000 to 174000
filled values from idx  174000 to 203000
filled values from idx  203000 to 232000
filled values from idx  232000 to 261000
filled values from idx  261000 to 290000
filled values from idx  290000 to 319000
filled values from idx  319000 to 348000
